# Processing text transcribed from spoken language

In [14]:
from pydub import AudioSegment
import speech_recognition as sr

import nltk
nltk.download('punkt')
nltk.download('vader_lexicon')

from nltk.sentiment.vader import SentimentIntensityAnalyzer
from nltk.tokenize import sent_tokenize

import spacy

/home/jorge/.pyenv/versions/natural-language-processing-in-python/lib/python3.13/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/jorge/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package punkt to /home/jorge/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/jorge/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


## Creating transcription helper functions

### Converting audio to the right format

In [2]:
# Create function to convert audio file to wav
def convert_to_wav(filename):
  """Takes an audio file of non .wav format and converts to .wav"""
  # Import audio file
  audio = AudioSegment.from_file(filename)
  
  # Create new filename
  new_filename = filename.split(".")[0] + ".wav"
  
  # Export file as .wav
  audio.export(new_filename, format='wav')
  print(f"Converting {filename} to {new_filename}...")
 
# Test the function
convert_to_wav('call_1.mp3')

Converting call_1.mp3 to call_1.wav...


### Finding PyDub stats

In [3]:
def show_pydub_stats(filename):
  """Returns different audio attributes related to an audio file."""
  # Create AudioSegment instance
  audio_segment = AudioSegment.from_file(filename)
  
  # Print audio attributes and return AudioSegment instance
  print(f"Channels: {audio_segment.channels}")
  print(f"Sample width: {audio_segment.sample_width}")
  print(f"Frame rate (sample rate): {audio_segment.frame_rate}")
  print(f"Frame width: {audio_segment.frame_width}")
  print(f"Length (ms): {len(audio_segment)}")
  return audio_segment

# Try the function
call_1_audio_segment = show_pydub_stats("call_1.wav")

Channels: 2
Sample width: 2
Frame rate (sample rate): 32000
Frame width: 4
Length (ms): 54888


### Transcribing audio with one line

In [4]:
def transcribe_audio(filename):
  """Takes a .wav format audio file and transcribes it to text."""
  # Setup a recognizer instance
  recognizer = sr.Recognizer()
  
  # Import the audio file and convert to audio data
  audio_file = sr.AudioFile(filename)
  with audio_file as source:
    audio_data = recognizer.record(source)
  
  # Return the transcribed text
  return recognizer.recognize_google(audio_data)

# Test the function
print(transcribe_audio("call_1.wav"))

hello welcome to Acme Studio support line my name is Daniel how can I best help you hey Danielle this is John I've recently bought a smartphone from you guys three weeks ago and I'm already having issues with it oh no that's not good to hear John let's let's get your serial number and then we can we can set up a way to fix it for you one second let me grab my Sarah number it is 41757 I'm very displeased how long do you reckon this is going to take me on hold for about an hour now John we're going to try out there I'm just I'm just really really really really really really really just I've been trying to contact support for the past past 3 for that really happy I want to get this issue fixed as fast as possible


### Using the helper functions you've built

In [5]:
# Convert mp3 file to wav
convert_to_wav("call_1.mp3")

# Check the stats of new file
call_1 = show_pydub_stats("call_1.wav")

# Split call_1 to mono
call_1_split = call_1.split_to_mono()

# Export channel 2 (the customer channel)
call_1_split[1].export("call_1_channel_2.wav", format="wav")

# Transcribe the single channel
print(transcribe_audio("call_1_channel_2.wav"))

Converting call_1.mp3 to call_1.wav...
Channels: 2
Sample width: 2
Frame rate (sample rate): 32000
Frame width: 4
Length (ms): 54888
hey Danielle this is John I've recently bought a smartphone from you guys 3 weeks ago and I'm already having issues with it is 417577


## Sentiment analysis on spoken language text

### Analyzing sentiment of a phone call

In [7]:
# Create SentimentIntensityAnalyzer instance
sid = SentimentIntensityAnalyzer()

# Let's try it on one of our phone calls
call_2_text = transcribe_audio('call_2.wav')

# Display text and sentiment polarity scores
print(call_2_text)
print(sid.polarity_scores(call_2_text))

hello my name is Daniel thank you for calling Acme Studios how can I best help you oh hi Daniel my name is Sally my recently purchased a smartphone from you guys and extremely happy with it I just got an issue not an issue but I just got to learn a little bit more about the message bank I have OK Google but I'm finding it hard I thought you were on the corner of Edward and Elizabeth according to Google according to the maps but would you be able to help me in some way because I think I actually walked straight past your shop yeah sure thing well thank you Sally that's good to hear you're enjoying it let me let me find out with the nearest stories for you
{'neg': 0.011, 'neu': 0.732, 'pos': 0.257, 'compound': 0.9882}


### Sentiment analysis on formatted text

In [10]:
# Create SentimentIntensityAnalyzer instance
sid = SentimentIntensityAnalyzer()

# Transcribe customer channel of call 2
call_2_channel_2_text = transcribe_audio('call_2_channel_2.wav')

# Display text and sentiment polarity scores
print(call_2_channel_2_text)
print(sid.polarity_scores(call_2_channel_2_text))

oh hi Daniel my name is Sally I recently purchased a smartphone from you guys and extremely happy with it I just got an issue not an issue but I've just got to learn a little bit more about the message bank I have Google the location but I'm finding it hard I thought you were on the corner of Edward and Elizabeth according to Google according to the maps but would you be able to help me in some way because I think I actually walked straight past your shop
{'neg': 0.018, 'neu': 0.888, 'pos': 0.094, 'compound': 0.778}


In [11]:
# Create SentimentIntensityAnalyzer instance
sid = SentimentIntensityAnalyzer()

# Split call 2 channel 2 into sentences and score each
for sentence in sent_tokenize(call_2_channel_2_text):
    print(sentence)
    print(sid.polarity_scores(sentence))

oh hi Daniel my name is Sally I recently purchased a smartphone from you guys and extremely happy with it I just got an issue not an issue but I've just got to learn a little bit more about the message bank I have Google the location but I'm finding it hard I thought you were on the corner of Edward and Elizabeth according to Google according to the maps but would you be able to help me in some way because I think I actually walked straight past your shop
{'neg': 0.018, 'neu': 0.888, 'pos': 0.094, 'compound': 0.778}


In [12]:
call_2_channel_2_paid_api_text = "Hello and welcome to acme studios. My name's Daniel. How can I best help you? Hi Diane. This is paid on this call up to see the status of my, I'm proctor mortars at three weeks ago, and then service is terrible. Okay, Peter, sorry to hear about that. Hey, Peter, before we go on, do you mind just, uh, is there something going on with your microphone? I can't quite hear you. Is this any better? Yeah, that's much better. And sorry, what was, what was it that you said when you first first started speaking?  So I ordered a product from you guys three weeks ago and, uh, it's, it's currently on July 1st and I haven't received a provocative, again, three weeks to a full four weeks down line. This service is terrible. Okay. Well, what's your order id? I'll, uh, I'll start looking into that for you. Six, nine, eight, seven five. Okay. Thank you."

# Create SentimentIntensityAnalyzer instance
sid = SentimentIntensityAnalyzer()

# Split channel 2 paid text into sentences and score each
for sentence in sent_tokenize(call_2_channel_2_paid_api_text):
    print(sentence)
    print(sid.polarity_scores(sentence))

Hello and welcome to acme studios.
{'neg': 0.0, 'neu': 0.625, 'pos': 0.375, 'compound': 0.4588}
My name's Daniel.
{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
How can I best help you?
{'neg': 0.0, 'neu': 0.303, 'pos': 0.697, 'compound': 0.7845}
Hi Diane.
{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
This is paid on this call up to see the status of my, I'm proctor mortars at three weeks ago, and then service is terrible.
{'neg': 0.114, 'neu': 0.886, 'pos': 0.0, 'compound': -0.4767}
Okay, Peter, sorry to hear about that.
{'neg': 0.159, 'neu': 0.61, 'pos': 0.232, 'compound': 0.1531}
Hey, Peter, before we go on, do you mind just, uh, is there something going on with your microphone?
{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
I can't quite hear you.
{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Is this any better?
{'neg': 0.0, 'neu': 0.508, 'pos': 0.492, 'compound': 0.4404}
Yeah, that's much better.
{'neg': 0.0, 'neu': 0.282, 'pos': 0.718, 'compound': 0.

## Named entity recognition on transcribed text

### Named entity recognition in spacy

In [15]:
# Transcribe call 4 channel 2
call_4_channel_2_text = transcribe_audio("call_4_channel_2.wav")

# Create a spaCy language model instance
nlp = spacy.load("en_core_web_sm")

# Create a spaCy doc with call 4 channel 2 text
doc = nlp(call_4_channel_2_text)

# Check the type of doc
print(type(doc))

<class 'spacy.tokens.doc.Doc'>


In [16]:
# Show tokens in doc
for token in doc:
    print(token.text, token.idx)

oh 0
hello 3
Daniel 9
my 16
name 19
is 24
Anne 27
and 32
I 36
've 37
recently 41
just 50
purchased 55
a 65
smartphone 67
from 78
you 83
and 87
I 91
'm 92
very 95
happy 100
with 106
the 111
product 115
I 123
'd 124
like 127
to 132
order 135
another 141
one 149
for 153
my 157
friend 160
who 167
lives 171
in 177
Sydney 180
and 187
have 191
it 196
delivered 199
I 209
'm 210
pretty 213
sure 220
it 225
's 227
model 230
3155 236
give 241
you 246
the 250
address 254
thank 262
you 268
excellent 272


In [17]:
# Show sentences in doc
for sentence in doc.sents:
    print(sentence)

oh hello Daniel my name is Anne
and I've recently just purchased a smartphone from you and I'm very happy with the product I'd like to order another one for my friend who lives in Sydney and have it delivered I'm pretty sure it's model 3155 give you the address thank you excellent


In [18]:
# Show named entities and their labels
for entity in doc.ents:
    print(entity.text, entity.label_)

Daniel PERSON
Anne PERSON
Sydney GPE
3155 DATE


### Creating a custom named entity in spacy

In [19]:
# Add an entity_ruler component to the pipeline, before ner
ruler = nlp.add_pipe("entity_ruler", before="ner")

# Define pattern for new entity
ruler.add_patterns([{"label": "PRODUCT", "pattern": 'smartphone'}])

# Recreate the doc so it uses the updated pipeline
doc = nlp(call_4_channel_2_text)

# Test new entity
for entity in doc.ents:
  print(entity.text, entity.label_)

Daniel PERSON
Anne PERSON
smartphone PRODUCT
Sydney GPE
3155 DATE


## Classifying transcribed speech with sklearn

### Preparing audio files for text classification

In [ ]:
# # Convert post purchase
# for file in post_purchase:
#     print(f"Converting {file} to .wav...")
#     convert_to_wav(file)

# # Convert pre purchase
# for file in pre_purchase:
#     print(f"Converting {file} to .wav...")
#     convert_to_wav(file)

### Transcribing phone call excerpts

In [ ]:
# def create_text_list(folder):
#   # Create empty list
#   text_list = []
  
#   # Go through each file
#   for file in folder:
#     # Make sure the file is .wav
#     if file.endswith(".wav"):
#       print(f"Transcribing file: {file}...")
      
#       # Transcribe audio and append text to list
#       text_list.append(transcribe_audio(file))   
#   return text_list

# create_text_list(folder)

# # Transcribe post and pre purchase text
# post_purchase_text = create_text_list(post_purchase_wav_files)
# pre_purchase_text = create_text_list(pre_purchase_wav_files)

# # Inspect the first transcription of post purchase
# print(post_purchase_text[0])

### Organizing transcribed phone call data

In [ ]:
# import pandas as pd

# # Make dataframes with the text
# post_purchase_df = pd.DataFrame({"label": "post_purchase",
#                                  "text": post_purchase_text})
# pre_purchase_df = pd.DataFrame({"label": "pre_purchase",
#                                 "text": pre_purchase_text})

# # Combine DataFrames
# df = pd.concat([post_purchase_df, pre_purchase_df])

# # Print the combined DataFrame
# print(df.head()

### Create a spoken language text clasifier

In [ ]:
# # Build the text_classifier as an sklearn pipeline
# text_classifier = Pipeline([
#     ('vectorizer', CountVectorizer()),
#     ('tfidf', TfidfTransformer()),
#     ('classifier', MultinomialNB()),
# ])

# # Fit the classifier pipeline on the training data
# text_classifier.fit(train_df.text, train_df.label)

# # Build the text_classifier as an sklearn pipeline
# text_classifier = Pipeline([
#     ('vectorizer', CountVectorizer()),
#     ('tfidf', TfidfTransformer()),
#     ('classifier', MultinomialNB()),
# ])

# # Fit the classifier pipeline on the training data
# text_classifier.fit(train_df.text, train_df.label)

# # Evaluate the MultinomialNB model
# predicted = text_classifier.predict(test_df.text)
# accuracy = 100 * np.mean(predicted == test_df.label)
# print(f'The model is {accuracy}% accurate')